In [ ]:
import os
os.getcwd()

In [ ]:
Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Code\Imports'

In [ ]:
os.chdir(Share_point)

In [ ]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

import eurostat  # python wrapper for taking data.
import time

In [ ]:
# Converter for metric tonnes to M3m 
converter = -0.001379

### In Million tonnes

In [ ]:
d1 = pd.read_excel('raw_data/LNG/Global LNG imports 2017-2018.xlsx')
d2 = pd.read_excel('raw_data/LNG/Global LNG imports 2019-2020.xlsx')
d3 = pd.read_excel('raw_data/LNG/Global LNG imports 2021-2022.xlsx')
d4 = pd.read_excel('raw_data/LNG/Global LNG imports 2023.xlsx')

In [ ]:
# Merge 2017 to 2020
df_20 = pd.merge(d1,d2,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2017 to 2022
df_22 = pd.merge(df_20,d3,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2022 to 2023
df = pd.merge(df_22,d4,on=['Origin_Berthcountry','Port_Country'],how='outer')

## little code to investigate total global flows

### Belgium transhipping checks

In [ ]:
# check who imports from BE
df[df['Origin_Berthcountry']=='Belgium'].iloc[:, :2].join(df[df['Origin_Berthcountry']=='Belgium'].iloc[:, -9:])

In [ ]:
lng_i = pd.DataFrame()
for country in ['Belgium','France','Spain']:
    lng_i['{}'.format(country)] = df[df['Port_Country']==country].sum(numeric_only=True)*converter*10.3/1000

In [ ]:
lng_i.to_excel(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2023-04 How the EU can phase out Russian LNG\Data\lng_BB.xlsx')

## EU27

In [ ]:
EU27 = ['Belgium', 'Croatia','Finland','France','Greece','Italy','Lithuania','Netherlands', 'Portugal','Spain','Sweden'] #, 'United Kingdom']

In [ ]:
eu = df[df['Port_Country'].isin(EU27)].reset_index()

In [ ]:
eu27 = eu.groupby('Origin_Berthcountry').sum()

In [ ]:
del eu27['index']

In [ ]:
eu27.columns=pd.to_datetime(eu27.columns)

In [ ]:
eu27_t = eu27.T

In [ ]:
eu27_t=eu27_t.sort_index()

In [ ]:
eu27_t.replace(0,np.nan,inplace=True)
df27=eu27_t.iloc[-9:,:].dropna(axis=1,how='all')

In [ ]:
# convert to TWh
df27=df27*converter*10.3/1000 

In [ ]:
df27.columns

In [ ]:
df27[['Russia','United States of America','Qatar','Norway','Trinidad & Tobago','Nigeria']]

### Single-out China

In [ ]:
china = df[df['Port_Country'].isin(['China'])].reset_index()

In [ ]:
cols = china.iloc[:,1:3]

In [ ]:
cols.tail()

In [ ]:
# convert to M3m
values = china.iloc[:,3:]*converter

In [ ]:
china = pd.concat([cols,values], axis=1)

In [ ]:
china = china.groupby(['Origin_Berthcountry']).sum(numeric_only=True)

In [ ]:
china = china.T #drop(

In [ ]:
china.index = pd.to_datetime(china.index)
china = china.sort_index()

In [ ]:
china.head()

In [ ]:
china.plot(figsize=(15, 9),kind='area',ylabel='M3m', title='LNG imports in China')

In [ ]:
col = china.columns

In [ ]:
china['Total'] = china.sum(axis=1)

In [ ]:
# percentage increase in chinese LNG imports
(china['Total']['2023-01':'2023-09'].sum()/china['Total']['2022-01':'2022-09'].sum()-1)*100

In [ ]:
china.columns

In [ ]:
## Define an "Other" category
china['Other']= china.loc[:, ~china.columns.isin(['Total','Russia','Australia','United States of America','Qatar','Indonesia','Malaysia'])].sum(axis=1)

In [ ]:
china_r = china[['Russia','Australia','Qatar','United States of America','Indonesia','Malaysia','Other']]

In [ ]:
china_twh = china_r*10.3/1000

In [ ]:
china_twh.plot(figsize=(15, 9),kind='area',ylabel='TWh', title='LNG imports in China')

## Single out Russia

In [ ]:
## Here we get 
russia = df[df['Origin_Berthcountry'].isin(['Russia'])].reset_index()

In [ ]:
cols = russia.iloc[:,1]

In [ ]:
values = russia.iloc[:,3:]*converter

In [ ]:
russia = pd.concat([cols,values], axis=1)

In [ ]:
russia = russia.groupby(['Port_Country']).sum(numeric_only=True)

In [ ]:
dfint = russia.iloc[:,-9:].replace(0, np.nan).dropna(how='all')

In [ ]:
dfint = dfint/1000*10.3 # convert in TWh

In [ ]:
dfint.to_excel(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data\LNG\Bloomberg\russia.xlsx')

In [ ]:
russia = russia.T

In [ ]:
russia.index = pd.to_datetime(russia.index)
russia = russia.sort_index()

In [ ]:
russia.columns

In [ ]:
russia_c = russia

In [ ]:
EU28 = ['Belgium', 'Croatia','Finland','France','Greece','Italy','Lithuania','Netherlands', 'Portugal','Spain','Sweden'] #, 'United Kingdom']
russia_c['EU28']= russia_c.loc[:, russia_c.columns.isin(EU28)].sum(axis=1)

In [ ]:
russia_c['RoW'] = russia_c.loc[:, ~russia_c.columns.isin(EU28+ ['China','EU28'])].sum(axis=1)

In [ ]:
russia_c.columns

In [ ]:
# EU imports from Russia in TWh 
russia_c['EU28']['2022'].sum()/1000*10.3

In [ ]:
# percentage going to the EU
russia_c['EU28']['2023'].sum()/(russia_c['RoW']['2023'].sum()+russia_c['EU28']['2023'].sum()+russia_c['China']['2023'].sum())*100

In [ ]:
russia_p = russia_c[['China', 'EU28','RoW']]

In [ ]:
russia_p.plot(figsize=(15, 9),kind='area',ylabel='M3m', title='LNG exports from Russia')

In [ ]:
russia_twh = russia_p/1000*10.3

In [ ]:
russia_twh.plot(figsize=(15, 9),kind='area',ylabel='TWh', title='LNG exports from Russia')

In [ ]:
import os
os.getcwd()

In [ ]:
russia_twh.to_excel(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2023-04 How the EU can phase out Russian LNG\Data\Russian exports 06.06.xlsx')